# Chapter 7 – Worked examples

This notebook runs the examples of [Chapter 7](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-2-circuits/ch07-electromagnetism/) live. The code lives in the modules next to it
(`coil_emf.py`, `conductor.py`); the cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q matplotlib numpy

In [ ]:
import coil_emf
import conductor
import matplotlib.pyplot as plt
import numpy as np

## Resistance and Joule losses of a transmission conductor

$R = \rho L / A$ for 100 km of 500 mm² aluminium, then $P = R I^2$ at 1 000 A, and the same 200 MW carried by
a two-conductor line at 200 kV and at 400 kV: the losses fall as $1/V^2$.

In [ ]:
print(f'R = {conductor.resistance():.2f} ohm, losses at 1000 A = {conductor.joule_losses(1000.0) / 1e6:.2f} MW')
for v in (200e3, 400e3):
    i, p = conductor.line_losses(200e6, v)
    print(f'{v / 1e3:.0f} kV: I = {i:.0f} A, losses = {p / 1e6:.2f} MW')

## A coil rotating in a uniform field

$N = 100$ turns, $A = 0.01\ \text{m}^2$, $B = 0.5$ T at 50 Hz: $\Phi(t) = NBA\cos\omega t$ and
$e(t) = -d\Phi/dt = NBA\omega \sin\omega t$. The numeric derivative (`np.gradient`) sits on top of the analytic
EMF; flux and EMF are 90° apart.

In [ ]:
t = coil_emf.sample_period(1000)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t * 1e3, coil_emf.flux(t) * 300, label='flux × 300 (Wb-turn)')
ax.plot(t * 1e3, coil_emf.emf_analytic(t), label='EMF, analytic (V)')
ax.plot(t * 1e3, coil_emf.emf_numeric(t), '--', label='EMF, np.gradient (V)')
ax.set_xlabel('t (ms)')
ax.legend()
plt.show()
print(f'peak EMF {coil_emf.E_PEAK:.2f} V, RMS {coil_emf.rms(coil_emf.emf_analytic(t)):.2f} V')

## RMS values and average power

The RMS of a sinusoid is its peak over $\sqrt 2$; the RMS values of a 50 Hz and a 150 Hz component add in
squares; the mean of $p(t) = e^2 / R$ equals $E_\text{RMS}^2 / R$.

In [ ]:
e1 = coil_emf.emf_analytic(t)
e3 = 40.0 * np.sin(3 * coil_emf.OMEGA * t)
r1, r3 = coil_emf.rms(e1), coil_emf.rms(e3)
print(f'RMS 50 Hz {r1:.3f} V, 150 Hz {r3:.3f} V, sum {coil_emf.rms(e1 + e3):.3f} V')
print(f'sqrt of the sum of squares {np.hypot(r1, r3):.3f} V')
print(f'mean power into 100 ohm {coil_emf.mean_power(e1, 100.0):.2f} W = E_rms^2 / R = {r1**2 / 100:.2f} W')
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t * 1e3, e1**2 / 100, label='p(t) = e² / R')
ax.axhline(coil_emf.mean_power(e1, 100.0), color='k', linestyle='--', label='mean = E_rms² / R')
ax.set_xlabel('t (ms)')
ax.set_ylabel('W')
ax.legend()
plt.show()